<a href="https://colab.research.google.com/github/Lalla-dev/ML-practice-projects/blob/main/fraud_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
train = pd.read_csv('/content/train.csv')
test =pd.read_csv('/content/test.csv')

In [ ]:
train.head()

,timestamp,user_id,processing_batch_id,transaction_amount,user_age_days,ip_risk_score,transaction_speed_seconds,product_category,payment_method,is_fraud
0,29:50.2,104985,2113,4.27,392,30.01,17.73,subscription,credit_card,0
1,22:20.7,102534,2254,37.62,507,25.83,22.36,travel,debit_card,0
2,46:36.0,110534,2373,531.63,64,85.22,9.35,digital_service,credit_card,1
3,38:02.3,109236,2158,314.96,965,55.09,22.81,subscription,paypal,0
4,22:58.1,109706,2381,2300.59,45,94.31,4.67,digital_service,credit_card,1


In [ ]:
test.head()

,timestamp,user_id,processing_batch_id,transaction_amount,user_age_days,ip_risk_score,transaction_speed_seconds,product_category,payment_method
0,34:19.5,100811,2362,1206.31,54,72.58,10.81,travel,credit_card
1,46:46.6,104345,2375,400.33,5,87.97,7.06,digital_service,paypal
2,27:23.4,100383,2339,109.19,740,54.21,11.82,subscription,credit_card
3,15:56.9,111014,2199,5.21,265,31.24,9.46,subscription,credit_card
4,47:51.0,110828,2282,11.17,379,38.05,20.80,apparel,debit_card


In [ ]:
test.describe()

,user_id,processing_batch_id,transaction_amount,user_age_days,ip_risk_score,transaction_speed_seconds
count,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000,2400.000000
mean,105942.007083,2246.729167,290.484375,376.972917,57.567125,13.160925
std,3520.290273,88.201350,542.050316,316.306183,25.318409,6.211895
min,100001.000000,2100.000000,0.380000,1.000000,10.030000,1.010000
25%,102804.500000,2171.000000,31.232500,69.000000,36.962500,8.237500
50%,105931.000000,2245.000000,96.230000,315.500000,60.295000,12.875000
75%,109033.750000,2323.000000,323.520000,656.000000,79.060000,18.190000
max,111999.000000,2399.000000,9964.400000,998.000000,99.970000,25.000000


In [ ]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9600 entries, 0 to 9599
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   timestamp                  9600 non-null   object 
 1   user_id                    9600 non-null   int64  
 2   processing_batch_id        9600 non-null   int64  
 3   transaction_amount         9600 non-null   float64
 4   user_age_days              9600 non-null   int64  
 5   ip_risk_score              9600 non-null   float64
 6   transaction_speed_seconds  9600 non-null   float64
 7   product_category           9600 non-null   object 
 8   payment_method             9600 non-null   object 
 9   is_fraud                   9600 non-null   int64  
dtypes: float64(3), int64(4), object(3)
memory usage: 750.1+ KB


In [ ]:
train.isnull().sum()

,0
timestamp,0
user_id,0
processing_batch_id,0
transaction_amount,0
user_age_days,0
ip_risk_score,0
transaction_speed_seconds,0
product_category,0
payment_method,0
is_fraud,0


In [ ]:
test.isnull().sum()

,0
timestamp,0
user_id,0
processing_batch_id,0
transaction_amount,0
user_age_days,0
ip_risk_score,0
transaction_speed_seconds,0
product_category,0
payment_method,0


In [ ]:
df = pd.concat([train, test], ignore_index = True )

In [ ]:
df

,timestamp,user_id,processing_batch_id,transaction_amount,user_age_days,ip_risk_score,transaction_speed_seconds,product_category,payment_method,is_fraud
0,29:50.2,104985,2113,4.27,392,30.01,17.73,subscription,credit_card,0.0
1,22:20.7,102534,2254,37.62,507,25.83,22.36,travel,debit_card,0.0
2,46:36.0,110534,2373,531.63,64,85.22,9.35,digital_service,credit_card,1.0
3,38:02.3,109236,2158,314.96,965,55.09,22.81,subscription,paypal,0.0
4,22:58.1,109706,2381,2300.59,45,94.31,4.67,digital_service,credit_card,1.0
...,...,...,...,...,...,...,...,...,...,...
11995,55:30.1,110834,2257,100.98,164,55.62,17.50,subscription,credit_card,NaN
11996,24:17.3,102130,2190,734.71,79,72.45,13.93,travel,credit_card,NaN
11997,26:11.7,109608,2336,17.50,718,67.66,13.14,digital_service,credit_card,NaN
11998,12:22.3,109950,2283,2373.22,754,74.81,21.17,apparel,paypal,NaN


In [ ]:
df.isnull().sum()

,0
timestamp,0
user_id,0
processing_batch_id,0
transaction_amount,0
user_age_days,0
ip_risk_score,0
transaction_speed_seconds,0
product_category,0
payment_method,0
is_fraud,2400


In [ ]:
df['product_category'].unique()

array(['subscription', 'travel', 'digital_service', 'apparel',
       'electronics'], dtype=object)

In [ ]:
df['product_category'] = df['product_category'].map({'subscription': 0, 'travel':1, 'digital_service':2, 'apparel':3, 'electronics':4})

In [ ]:
df['payment_method'].unique()

array(['credit_card', 'debit_card', 'paypal'], dtype=object)

In [ ]:
df['payment_method'] = df['payment_method'].map({'credit_card': 0, 'debit_card':1, 'paypal':2})

In [ ]:
df['is_fraud'].isnull().sum()

np.int64(2400)

In [ ]:
df['is_fraud'].unique()

array([ 0.,  1., nan])

In [ ]:
print("TRAIN SHAPE:", train.shape)
print("TEST SHAPE :", test.shape)

print("\nUnique processing batches:")
print("Train:", train['processing_batch_id'].nunique())
print("Test :", test['processing_batch_id'].nunique())
print("Combined:", pd.concat([
    train['processing_batch_id'],
    test['processing_batch_id']
]).nunique())

print("\nUnique users:")
print("Train:", train['user_id'].nunique())
print("Test :", test['user_id'].nunique())
print("Combined:", pd.concat([
    train['user_id'],
    test['user_id']
]).nunique())

TRAIN SHAPE: (9600, 10)
TEST SHAPE : (2400, 9)

Unique processing batches:
Train: 300
Test : 300
Combined: 300

Unique users:
Train: 9600
Test : 2400
Combined: 12000


In [ ]:
df

,timestamp,user_id,processing_batch_id,transaction_amount,user_age_days,ip_risk_score,transaction_speed_seconds,product_category,payment_method,is_fraud
0,29:50.2,104985,2113,4.27,392,30.01,17.73,0,0,0.0
1,22:20.7,102534,2254,37.62,507,25.83,22.36,1,1,0.0
2,46:36.0,110534,2373,531.63,64,85.22,9.35,2,0,1.0
3,38:02.3,109236,2158,314.96,965,55.09,22.81,0,2,0.0
4,22:58.1,109706,2381,2300.59,45,94.31,4.67,2,0,1.0
...,...,...,...,...,...,...,...,...,...,...
11995,55:30.1,110834,2257,100.98,164,55.62,17.50,0,0,NaN
11996,24:17.3,102130,2190,734.71,79,72.45,13.93,1,0,NaN
11997,26:11.7,109608,2336,17.50,718,67.66,13.14,2,0,NaN
11998,12:22.3,109950,2283,2373.22,754,74.81,21.17,3,2,NaN


In [ ]:
df = pd.get_dummies(
    df,
    columns=['product_category', 'payment_method'],
    dtype=int
)

df.head()

,timestamp,user_id,processing_batch_id,transaction_amount,user_age_days,ip_risk_score,transaction_speed_seconds,is_fraud,product_category_0,product_category_1,product_category_2,product_category_3,product_category_4,payment_method_0,payment_method_1,payment_method_2
0,29:50.2,104985,2113,4.27,392,30.01,17.73,0.0,1,0,0,0,0,1,0,0
1,22:20.7,102534,2254,37.62,507,25.83,22.36,0.0,0,1,0,0,0,0,1,0
2,46:36.0,110534,2373,531.63,64,85.22,9.35,1.0,0,0,1,0,0,1,0,0
3,38:02.3,109236,2158,314.96,965,55.09,22.81,0.0,1,0,0,0,0,0,0,1
4,22:58.1,109706,2381,2300.59,45,94.31,4.67,1.0,0,0,1,0,0,1,0,0


In [ ]:
df

,timestamp,user_id,processing_batch_id,transaction_amount,user_age_days,ip_risk_score,transaction_speed_seconds,is_fraud,product_category_0,product_category_1,product_category_2,product_category_3,product_category_4,payment_method_0,payment_method_1,payment_method_2
0,29:50.2,104985,2113,4.27,392,30.01,17.73,0.0,1,0,0,0,0,1,0,0
1,22:20.7,102534,2254,37.62,507,25.83,22.36,0.0,0,1,0,0,0,0,1,0
2,46:36.0,110534,2373,531.63,64,85.22,9.35,1.0,0,0,1,0,0,1,0,0
3,38:02.3,109236,2158,314.96,965,55.09,22.81,0.0,1,0,0,0,0,0,0,1
4,22:58.1,109706,2381,2300.59,45,94.31,4.67,1.0,0,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11995,55:30.1,110834,2257,100.98,164,55.62,17.50,NaN,1,0,0,0,0,1,0,0
11996,24:17.3,102130,2190,734.71,79,72.45,13.93,NaN,0,1,0,0,0,1,0,0
11997,26:11.7,109608,2336,17.50,718,67.66,13.14,NaN,0,0,1,0,0,1,0,0
11998,12:22.3,109950,2283,2373.22,754,74.81,21.17,NaN,0,0,0,1,0,0,0,1


In [ ]:
dfgrp= df.groupby('processing_batch_id')['is_fraud'].agg(
    transactions = 'sum',
    fraud_rate = 'mean',
    transaction_speed_seconds = 'mean',
    frauds = 'sum'
)

In [ ]:

dfgrp.sort_values(by='frauds', ascending=False).head(10)

,transactions,fraud_rate,transaction_speed_seconds,frauds
processing_batch_id,,,,
2108,15.0,0.405405,0.405405,15.0
2166,15.0,0.312500,0.312500,15.0
2355,15.0,0.357143,0.357143,15.0
2206,15.0,0.357143,0.357143,15.0
2347,15.0,0.416667,0.416667,15.0
2130,14.0,0.482759,0.482759,14.0
2161,14.0,0.388889,0.388889,14.0
2243,14.0,0.400000,0.400000,14.0
2238,14.0,0.388889,0.388889,14.0


In [ ]:
dfgrp['transaction_speed_seconds'].head(5).mean()


np.float64(0.27572843822843823)

In [ ]:
dfgrp['transaction_speed_seconds'].head(10).mean()


np.float64(0.300470899070037)

In [ ]:
dfgrp['transaction_speed_seconds'].head(15).mean()


np.float64(0.29857255340301314)

In [ ]:
dfgrp['transaction_speed_seconds'].head(20).mean()


np.float64(0.28399884599210234)

In [ ]:
dfgrp['transaction_speed_seconds'].head(25).mean()


np.float64(0.2734517083726293)

In [ ]:
dfgrp['transaction_speed_seconds'].head(30).mean()

np.float64(0.2716264048454518)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
model_df= df.copy()

In [ ]:
time_parts = model_df['timestamp'].str.split(":", expand=True)

In [ ]:
model_df['timestamp_seconds'] = time_parts[0].astype(float) * 60 + time_parts[1].astype(float)

In [ ]:
df

,timestamp,user_id,processing_batch_id,transaction_amount,user_age_days,ip_risk_score,transaction_speed_seconds,is_fraud,product_category_0,product_category_1,product_category_2,product_category_3,product_category_4,payment_method_0,payment_method_1,payment_method_2
0,29:50.2,104985,2113,4.27,392,30.01,17.73,0.0,1,0,0,0,0,1,0,0
1,22:20.7,102534,2254,37.62,507,25.83,22.36,0.0,0,1,0,0,0,0,1,0
2,46:36.0,110534,2373,531.63,64,85.22,9.35,1.0,0,0,1,0,0,1,0,0
3,38:02.3,109236,2158,314.96,965,55.09,22.81,0.0,1,0,0,0,0,0,0,1
4,22:58.1,109706,2381,2300.59,45,94.31,4.67,1.0,0,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11995,55:30.1,110834,2257,100.98,164,55.62,17.50,NaN,1,0,0,0,0,1,0,0
11996,24:17.3,102130,2190,734.71,79,72.45,13.93,NaN,0,1,0,0,0,1,0,0
11997,26:11.7,109608,2336,17.50,718,67.66,13.14,NaN,0,0,1,0,0,1,0,0
11998,12:22.3,109950,2283,2373.22,754,74.81,21.17,NaN,0,0,0,1,0,0,0,1


In [ ]:
model_df = model_df.drop(columns = ['timestamp','user_id','product_category_apparel', 'product_category_digital_service','product_category_electronics','product_category_subscription','product_category_travel','payment_method_credit_card','payment_method_debit_card','payment_method_paypal'])

KeyError: "['product_category_apparel', 'product_category_digital_service', 'product_category_electronics', 'product_category_subscription', 'product_category_travel', 'payment_method_credit_card', 'payment_method_debit_card', 'payment_method_paypal'] not found in axis"

In [ ]:
model_df = model_df.drop(columns=['product_category_apparel', 'product_category_digital_service','product_category_electronics','product_category_subscription','product_category_travel','payment_method_credit_card','payment_method_debit_card','payment_method_paypal'])

In [ ]:
train_model = model_df.iloc[:len(train)].copy()
test_model = model_df.iloc[:len(train)].copy()

In [ ]:
x=train_model.drop(columns=['is_fraud'])
y=train_model['is_fraud']

In [ ]:
x_test = test_model.drop(columns=['is_fraud'])

print("X:", x.shape)
print("y:", y.shape)
print("X_test:", x_test.shape)

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(
    x,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training:", x_train.shape)
print("Validation:", x_val.shape)

In [ ]:
model = ExtraTreesClassifier(
    n_estimators=500,
    max_features='sqrt',
    min_samples_leaf=1,
    class_weight=None,
    random_state=42,
    n_jobs=-1
)

model.fit(x_train, y_train)

val_pred = model.predict(x_val)

accuracy = accuracy_score(y_val, val_pred)

print("Validation Accuracy:", accuracy)
print()
print(classification_report(y_val, val_pred))

In [ ]:
importance = pd.Series(
    model.feature_importances_,
    index=x.columns
).sort_values(ascending=False)

display(importance.head(15))

In [ ]:
final_model = ExtraTreesClassifier(
    n_estimators=800,
    max_features='sqrt',
    min_samples_leaf=1,
    class_weight=None,
    random_state=42,
    n_jobs=-1
)

final_model.fit(x, y)

test_pred = final_model.predict(X_test)

print("Predicted fraud:", test_pred.sum())
print("Predicted legitimate:", (test_pred == 0).sum())

In [ ]:
X = df.iloc[:len(train)].drop(columns=['is_fraud', 'user_id'])
y = train['is_fraud']

X_test = df.iloc[len(train):].drop(columns=['is_fraud', 'user_id'])

print(X.shape)
print(y.shape)
print(X_test.shape)

In [ ]:
submission = pd.DataFrame({
    'user_id': test['user_id'].values,
    'is_fraud': test_pred.astype(int)
})

submission.head(10)

In [ ]:
print("X:", x.shape)
print("X_test:", x_test.shape)
print("test rows:", len(test))
print("test_pred:", len(test_pred))

In [ ]:
final_model = ExtraTreesClassifier(
    n_estimators=800,
    max_features='sqrt',
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

final_model.fit(x, y)

print("Final model trained!")

In [ ]:
test_pred = final_model.predict(x_test)

print("Prediction length:", len(test_pred))
print("Fraud predictions:", test_pred.sum())
print("Legitimate predictions:", (test_pred == 0).sum())

In [ ]:
submission.to_csv('submission.csv', index=False)

print("✅ submission.csv created!")